# NUTMEG: iNsights from single cell mUltioMics with interprEtable GNNs

NUTMEG is a framework that turns single-cell multiomic data into gene regulatory networks and uses graph neural networks (GNNs) to predict cell states in an interpretable way. It is designed to help you move from raw counts and peak matrices to mechanistic hypotheses about which transcription factors (TFs), cis‑regulatory elements (CREs), and target genes drive each cell type.

## What NUTMEG does

- **Builds a global regulatory scaffold**  
  Combines TF–CRE motif edges and CRE–Gene links into a base gene regulatory network that is shared across all cells.

- **Creates per‑cell GRN graphs**  
  For each cell, attaches cell‑specific TF expression, gene expression, and CRE accessibility to the base network and prunes inactive parts of the graph.

- **Trains an interpretable GNN**  
  Learns to classify cells (e.g. cell types or states) directly on these heterogeneous graphs using a NUTMEG model based on GNN layers.

- **Explains its predictions**  
  Identifies important TF → CRE → Gene paths and scores them, so you can see which regulatory programs are most associated with each predicted cell type.

### Initialize NUTMEG dataset manager

Create a `GRNDatasetManager` to define where the input data live on disk, which analysis mode to use, and where to write output plots.  
This object relies on a `GRNDatasetPaths` configuration and will be used by all downstream steps to load features and build GRN graphs.

#### Arguments

- `experiment_dir`  
  Root directory for the current experiment. This folder should contain:
  - The feature matrices and feature-name files for genes (and CREs, if using `"multiome"`).
  - The cell label file with barcodes and cell type annotations.
  - Any additional files needed to define the GRN for this experiment.

- `plots_dir`  
  Directory where all plots and QC summaries will be saved.  

In [1]:
from dataset import GRNDatasetPaths, GRNDatasetManager

paths = GRNDatasetPaths(
    experiment_dir = "pbmc_rna_only_02",
    plots_dir="pbmc_rna_only_02/plots",
)
mgr = GRNDatasetManager(paths)


/home/fotini/miniconda3/envs/multiome_env/lib/python3.14/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._show_config()
/home/fotini/miniconda3/envs/multiome_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fotini/miniconda3/envs/multiome_env/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.0.post2)/ch

### Build the global base GRN

Construct a **base gene regulatory network (GRN)** that defines the shared TF–CRE–Gene scaffold used for all cells in the dataset.  
This network is built once and reused when creating the per‑cell graphs.

#### Arguments

- `mode`  
  Select the data modality used to define the GRN. Allowed values are:
  - `"multiome"`: build the GRN using information from both RNA and ATAC modalities.
  - `"rna_only"`: build the GRN using RNA‑based information only.

The resulting base GRN includes:

- **Gene nodes** representing transcription factors and target genes.  
- **CRE nodes** representing promoters and distal regulatory elements.  
- **Directed edges** for TF → CRE (binding) and CRE → Gene (regulation).


In [2]:
mgr.build_base_grn(mode="rna_only")

[BaseGRN] Loaded:
  - mode:          rna_only
  - tf_peak_scores (351937, 4)
  - peaks_genes    (14986, 5)

[BaseGRN] Unified nodes (rna_only):
  - TFs:                 59
  - Targets (genes):     1770
  - TF ∩ Target:         59
  - TF only:             0
  - Target only:         1711
  - Genes (TF ∪ Target): 1770
[BaseGRN] Degree plot saved -> /home/fotini/Desktop/lala/pbmc_rna_only_02/plots/base_grn/base_grn_degree_distributions.png


HeteroData(
  gene={
    num_nodes=1770,
    names=[1770],
  },
  (gene, regulates, gene)={
    edge_index=[2, 73154],
    edge_attr=[73154, 2],
  }
)

###  Automatic Feature Matrix Discovery

The feature loader:
- Automatically discovers all required matrices
- Verifies dimensions
- Ensures all modalities share the same cell barcodes


In [3]:
mgr.load_feature_dir()

[discover] Found (mode=rna_only):
  - barcodes_file: barcodes.txt
  - tf_features: tf_features.txt
  - gene_features: gene_features.txt
  - tf_raw_mtx: tf_raw.mtx
  - gene_raw_mtx: gene_raw.mtx
  - tf_sct_mtx: tf_sct.mtx
  - gene_sct_mtx: gene_sct.mtx
[load] tf_feat      shape=(59, 2484)
[load] gene_feat    shape=(1770, 2484)
[load] tf_raw       shape=(59, 2484)
[load] gene_raw     shape=(1770, 2484)
[Feature] Loaded matrices for 2484 barcodes (from barcodes.txt)


### Inspect Cell Type Distribution

Before building per-cell graphs, we inspect the **cell type distribution** after intersecting the feature matrices with the label annotations.

This step helps determine:
- min_cells_per_class
- max_cells_per_class


In [4]:
mgr.inspect_cell_distribution(save_plot=True)


[Cell distribution] (after intersect with feature barcodes)


/home/fotini/Desktop/lala/dataset.py:1021: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(counts.index.tolist(), rotation=45, ha="right")


[inspect_plot] Saved -> /home/fotini/Desktop/lala/pbmc_rna_only_02/plots/dataset/cell_type_distribution.png


label
CD14 Mono         825
CD8 Naive         319
CD4 TCM           310
CD4 Naive         293
CD8 TEM           163
B naive           122
CD16 Mono         104
B intermediate     93
NK                 91
cDC2               31
MAIT               27
pDC                26
Treg               26
B memory           20
CD4 TEM            13
gdT                10
CD4 CTL             3
Plasmablast         3
CD8 TCM             3
HSPC                2
Name: count, dtype: int64

### Construct per‑cell GRN graphs

Create a graph‑based dataset where **each cell** is represented as a heterogeneous GRN.  
This step starts from the global base GRN and attaches cell‑specific features, pruning inactive parts of the network and filtering cell types by abundance.

#### Arguments

- `prune`  
  Control whether to remove inactive parts of the GRN for each cell.
  - Default: `TRUE`
  - `TRUE`: prune genes, CREs, and edges that are inactive in that cell, making graphs smaller and more specific.
  - `FALSE`: keep the full base GRN structure for every cell, regardless of activity.


- `min_cells_per_class`  
  Minimum number of cells required for a cell type to be kept in the dataset.


- `max_cells_per_class`  
  Maximum number of cells to keep per cell type.


The output is a `dataset` object where each item is one cell‑specific GRN graph with:
- TF, gene, and CRE nodes carrying cell‑specific features.
- Edges representing TF → CRE and CRE → Gene regulatory relations.
- A graph‑level label corresponding to the cell type.

In [5]:
dataset = mgr.build_per_cell_dataset(min_cells_per_class=70, max_cells_per_class = None)


[Labels] Pruned to 9 classes; remapped to contiguous IDs 0..8
[Labels] (Info) Received a global label_to_id with 20 entries; kept only surviving labels and rebuilt a compact mapping for training.
Kept 2320 cells 


### Summarize GRN dataset

Generates a high‑level summary and quality control report for the per‑cell GRN dataset.  
This step checks basic statistics of the graphs and saves diagnostic plots to help you verify that the constructed graphs look reasonable.


Computes dataset‑level statistics, including:
  - Total number of cells / graphs.
  - Average numbers of TF, gene, CRE nodes per graph.
  - Average numbers of TF → CRE and CRE → Gene edges.
  - Counts of autoregulatory TF loops and feature dimensions for each node type.



In [6]:
mgr.summarize_dataset(dataset)


Dataset Summary
Total cells/graphs: 2320
Plots will be saved in: /home/fotini/Desktop/lala/pbmc_rna_only_02/plots/dataset_summary

[Labels] Distribution (over summarized graphs):
CD14 Mono         825
CD8 Naive         319
CD4 TCM           310
CD4 Naive         293
CD8 TEM           163
B naive           122
CD16 Mono         104
B intermediate     93
NK                 91
Name: count, dtype: int64
[saved] pbmc_rna_only_02/plots/dataset_summary/label_distribution.png
[saved] pbmc_rna_only_02/plots/dataset_summary/gene_nodes_hist.png
[saved] pbmc_rna_only_02/plots/dataset_summary/tf_nodes_hist.png
[saved] pbmc_rna_only_02/plots/dataset_summary/target_nodes_hist.png
[saved] pbmc_rna_only_02/plots/dataset_summary/autoreg_tfs_hist.png
[saved] pbmc_rna_only_02/plots/dataset_summary/gene_gene_edges_hist.png

[Stats] Averages over summarized graphs:
  Avg gene nodes:          300.8
  Avg Gene→Gene edges:     3288.4
  Avg regulator nodes:     14.2
  Avg target nodes:        300.6

[Dims] Fea

### Split dataset into training and test sets

Partition the per‑cell GRN dataset into separate subsets used for model training and final evaluation.  

#### Arguments

- `train_frac`  
  Fraction of cells/graphs assigned to the training subset.

- `test_frac`  
  Fraction of cells/graphs assigned to the test subset.

- `seed`  
  Random seed used for shuffling before splitting.


In [7]:
train_ds, test_ds = dataset.split(train_frac=0.8, test_frac=0.2, seed=42)

Training subset:  1856 
Testing subset:  464


### Configure and train the NUTMEG model

Set up a `NUTMEGTrainer` to train a NUTMEG graph neural network on the per‑cell GRN dataset and evaluate its performance.  
The trainer reads model and optimization settings from a JSON file and manages the full training loop, validation, and test evaluation.

#### Arguments

- `json_path`  
  Path to a JSON file containing model architecture and training hyperparameters. 

- `dataset`  
  The full GRN dataset returned by `mgr.build_per_cell_dataset(...)`.  

- `model_ctor`  
  Constructor for the model class to train.  
  Here we pass `NUTMEG`, which builds a NUTMEG GNN configured according to `model_params.json`.

- `run_name`  
  Short identifier for this training run.

- `results_root`  
  Root directory where all run‑specific outputs are written.

In [7]:
from model import NUTMEG
from trainer import NUTMEGTrainer 


runner = NUTMEGTrainer.initialize(
    json_path="model_params.json",
    dataset=dataset,
    model_ctor=NUTMEG,
    run_name="exp1",
    results_root= "pbmc_rna_only_02"
)

fit_out = runner.train(train_ds)   
test_out = runner.evaluate(test_ds)



ModuleNotFoundError: No module named 'model'

In [13]:
from inter import NUTMEGResults

res = NUTMEGResults.from_runner(runner, test_ds, name="test")

out = res.interpret(
    runner,
    test_ds,
    mode="pred",
    top_k=10,
    top_k_per_cell=50,
    min_cells=3,
    path_score="geometric_mean",
    contributor_methods=("score_weight", "linear_shap"),
    contrastive=True,
    background="not_class",
    background_mode="pred",
    balance_background=True,
    n_permutations=1000,
    q_value_threshold=0.05,
    random_seed=42,
)


KeyboardInterrupt: 